# Problem B: Synthetic Event Log Evaluation (Fidelity · Utility · Privacy)

This notebook follows the provided guideline exactly at the framework level. It evaluates three synthetic log generators using:

- **Fidelity**: activity frequency KL-divergence, trace variant Jaccard similarity, case-duration Wasserstein distance
- **Utility**: TSTR protocol for three clinical prediction targets: readmission, LOS exceedance, admission conversion
- **Privacy**: ML-Leaks-style membership inference with a shadow generator and MLP attack classifier

Important validity rule: the notebook does **not** fabricate clinical labels when a synthetic log lacks the required fields. Such cells are reported as `NA` with a diagnostic reason.

In [ ]:
# ============================================================
# 0. Imports
# ============================================================
import os, re, json, math, zipfile, warnings, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy.special import rel_entr
from scipy.stats import wasserstein_distance

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('xgboost available:', HAS_XGB)

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
DATA_ZIP = Path('./data.zip')
DATA_DIR = Path('./data')
OUT_DIR = Path('./results_problemB_evaluation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COL = 'stay_id'
ACT_COL = 'activity'
TIME_COL = 'timestamps'

TARGETS = {
    'readmission_30d': '재입원 예측',
    'los_over_4h': '체류시간 초과',
    'admission_conversion': '입원 전환',
}

PREFIX_EVENTS = 3              # early prefix features only; prevents direct outcome leakage
LOS_THRESHOLD_HOURS = 4.0
READMISSION_WINDOW_DAYS = 30
N_SHADOW_SYNTH_CASES = 1000
N_ATTACK_MAX_PER_CLASS = 2000

print('OUT_DIR:', OUT_DIR.resolve())

In [ ]:
# ============================================================
# 2. Extract and discover files
# ============================================================
if not DATA_DIR.exists():
    if DATA_ZIP.exists():
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(DATA_DIR)
        print(f'Extracted {DATA_ZIP} -> {DATA_DIR}')
    else:
        raise FileNotFoundError('data.zip not found and ./data directory does not exist.')
else:
    print('Using existing ./data directory')

print('CSV files:')
for p in sorted(DATA_DIR.rglob('*.csv')):
    print(' -', p)

In [ ]:
# ============================================================
# 3. Loading and standardization
# ============================================================
def standardize_event_log(df: pd.DataFrame, source_name: str = '') -> pd.DataFrame:
    df = df.copy()
    rename = {}
    if 'case:concept:name' in df.columns and CASE_COL not in df.columns:
        rename['case:concept:name'] = CASE_COL
    if 'concept:name' in df.columns and ACT_COL not in df.columns:
        rename['concept:name'] = ACT_COL
    if 'time:timestamp' in df.columns and TIME_COL not in df.columns:
        rename['time:timestamp'] = TIME_COL
    if 'timestamp' in df.columns and TIME_COL not in df.columns:
        rename['timestamp'] = TIME_COL
    if 'case_id' in df.columns and CASE_COL not in df.columns:
        rename['case_id'] = CASE_COL
    df = df.rename(columns=rename)
    missing = [c for c in [CASE_COL, ACT_COL, TIME_COL] if c not in df.columns]
    if missing:
        raise ValueError(f'{source_name}: missing required columns {missing}; columns={df.columns.tolist()}')
    df[CASE_COL] = df[CASE_COL].astype(str)
    df[ACT_COL] = df[ACT_COL].astype(str).fillna('MISSING')
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True).dt.tz_convert(None)
    if 'event_order' in df.columns:
        df = df.sort_values([CASE_COL, 'event_order', TIME_COL], kind='mergesort')
    else:
        df = df.sort_values([CASE_COL, TIME_COL], kind='mergesort')
    return df.reset_index(drop=True)

def load_csv(path: Path, source_name=''):
    return standardize_event_log(pd.read_csv(path), source_name or str(path))

real_train = load_csv(DATA_DIR/'Real'/'mimicel_train.csv', 'real_train')
real_test = load_csv(DATA_DIR/'Real'/'mimicel_test.csv', 'real_test')
real_val_path = DATA_DIR/'Real'/'mimicel_val.csv'
real_val = load_csv(real_val_path, 'real_val') if real_val_path.exists() else None

print('real_train:', real_train.shape, 'cases:', real_train[CASE_COL].nunique())
print('real_test :', real_test.shape, 'cases:', real_test[CASE_COL].nunique())
if real_val is not None:
    print('real_val  :', real_val.shape, 'cases:', real_val[CASE_COL].nunique())
print('real columns:', real_train.columns.tolist())

In [ ]:
# ============================================================
# 4. Load synthetic logs
# ============================================================
def discover_synthetic_logs(data_dir: Path):
    rows = []
    folders = {'ProcessGAN': data_dir/'ProcessGAN', 'PALSYN': data_dir/'PALSYN', 'Rule_based': data_dir/'Rule_based'}
    for gen, folder in folders.items():
        if not folder.exists():
            print('[WARN] missing folder:', folder)
            continue
        for path in sorted(folder.glob('*.csv')):
            m = re.search(r'(?:seed)?(\d+)', path.name, flags=re.I)
            seed = int(m.group(1)) if m else len(rows)
            rows.append({'generator': gen, 'seed': seed, 'path': str(path)})
    return pd.DataFrame(rows)

syn_specs = discover_synthetic_logs(DATA_DIR)
display(syn_specs)

synthetic_logs = {}
for _, row in syn_specs.iterrows():
    key = (row['generator'], int(row['seed']))
    try:
        synthetic_logs[key] = load_csv(Path(row['path']), f'{row["generator"]}_{row["seed"]}')
        print(key, synthetic_logs[key].shape, 'cases:', synthetic_logs[key][CASE_COL].nunique())
    except Exception as e:
        print('[LOAD FAILED]', key, e)

In [ ]:
# ============================================================
# 5. Case-level labels and prefix features
# ============================================================
def _safe_mode(s):
    s = s.dropna()
    if len(s) == 0:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]

def build_case_table(df: pd.DataFrame, dataset_name=''):
    rows = []
    has_subject = 'subject_id' in df.columns
    has_hadm = 'hadm_id' in df.columns
    has_disposition = 'disposition' in df.columns
    for cid, g in df.groupby(CASE_COL, sort=False):
        g = g.sort_values(TIME_COL)
        acts = g[ACT_COL].astype(str).tolist()
        times = g[TIME_COL]
        start, end = times.min(), times.max()
        duration_hours = (end-start).total_seconds()/3600.0 if pd.notna(start) and pd.notna(end) else np.nan
        prefix_acts = acts[:PREFIX_EVENTS]
        prefix_times = times.iloc[:PREFIX_EVENTS]
        prefix_elapsed = (prefix_times.max()-prefix_times.min()).total_seconds()/3600.0 if len(prefix_times)>1 and prefix_times.notna().all() else 0.0
        row = {
            CASE_COL: cid, 'dataset': dataset_name,
            'n_events_full': len(acts), 'duration_hours_full': duration_hours,
            'trace_variant': '>>'.join(acts),
            'first_activity': acts[0] if acts else 'MISSING',
            'last_activity': acts[-1] if acts else 'MISSING',
            'prefix_len': len(prefix_acts), 'prefix_elapsed_hours': prefix_elapsed,
            'prefix_variant': '>>'.join(prefix_acts) if prefix_acts else 'PAD',
            'case_start': start, 'case_end': end,
        }
        for i in range(PREFIX_EVENTS):
            row[f'prefix_act_{i+1}'] = prefix_acts[i] if i < len(prefix_acts) else 'PAD'
        for col in ['gender','race','arrival_transport','acuity','chiefcomplaint']:
            if col in df.columns:
                row[col] = _safe_mode(g[col])
        row['los_over_4h'] = int(duration_hours > LOS_THRESHOLD_HOURS) if pd.notna(duration_hours) else np.nan
        if has_hadm:
            row['admission_conversion'] = int(g['hadm_id'].notna().any())
        elif has_disposition:
            disp = ' '.join(g['disposition'].dropna().astype(str).str.upper().unique())
            admit_terms = ['ADMIT','ADMITTED','HOSPITAL','INPATIENT','OBSERVATION']
            row['admission_conversion'] = int(any(t in disp for t in admit_terms)) if disp else np.nan
        else:
            row['admission_conversion'] = np.nan
        row['subject_id'] = _safe_mode(g['subject_id']) if has_subject else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def add_readmission_label(case_df):
    cdf = case_df.copy()
    cdf['readmission_30d'] = np.nan
    if 'subject_id' not in cdf.columns or cdf['subject_id'].isna().all():
        return cdf
    tmp = cdf.dropna(subset=['subject_id','case_start','case_end'])
    for sid, g in tmp.groupby('subject_id'):
        g = g.sort_values('case_start')
        idxs = g.index.tolist(); starts = g['case_start'].tolist(); ends = g['case_end'].tolist()
        for pos, idx in enumerate(idxs):
            label = 0
            for nxt in range(pos+1, len(idxs)):
                delta = starts[nxt] - ends[pos]
                if pd.Timedelta(0) <= delta <= pd.Timedelta(days=READMISSION_WINDOW_DAYS):
                    label = 1; break
                if delta > pd.Timedelta(days=READMISSION_WINDOW_DAYS):
                    break
            cdf.loc[idx, 'readmission_30d'] = label
    return cdf

real_train_cases = add_readmission_label(build_case_table(real_train, 'real_train'))
real_test_cases = add_readmission_label(build_case_table(real_test, 'real_test'))
real_val_cases = add_readmission_label(build_case_table(real_val, 'real_val')) if real_val is not None else None
print('Real label summary')
display(real_train_cases[list(TARGETS)].agg(['count','mean']))

syn_case_tables = {}
for key, df in synthetic_logs.items():
    cdf = add_readmission_label(build_case_table(df, f'{key[0]}_seed{key[1]}'))
    syn_case_tables[key] = cdf
    print('Synthetic label summary:', key, 'cases=', len(cdf))
    display(cdf[list(TARGETS)].agg(['count','mean']))

In [ ]:
# ============================================================
# 6. Fidelity metrics
# ============================================================
def activity_distribution(df):
    return df[ACT_COL].astype(str).value_counts(normalize=True)

def activity_kl(real_df, syn_df, eps=1e-9):
    p, q = activity_distribution(real_df), activity_distribution(syn_df)
    acts = sorted(set(p.index) | set(q.index))
    pv = np.array([p.get(a,0.0) for a in acts]) + eps
    qv = np.array([q.get(a,0.0) for a in acts]) + eps
    pv, qv = pv/pv.sum(), qv/qv.sum()
    return float(np.sum(rel_entr(pv, qv)))

def trace_variants(df):
    return {tuple(g[ACT_COL].astype(str).tolist()) for _, g in df.sort_values([CASE_COL,TIME_COL]).groupby(CASE_COL)}

def trace_jaccard(real_df, syn_df):
    a, b = trace_variants(real_df), trace_variants(syn_df)
    return float(len(a & b) / len(a | b)) if len(a | b) else np.nan

def durations(df):
    vals=[]
    for _, g in df.groupby(CASE_COL):
        t = pd.to_datetime(g[TIME_COL], errors='coerce')
        if t.notna().sum() >= 2:
            vals.append((t.max()-t.min()).total_seconds()/3600.0)
    return np.array(vals, dtype=float)

def duration_wasserstein(real_df, syn_df):
    r, s = durations(real_df), durations(syn_df)
    return float(wasserstein_distance(r,s)) if len(r) and len(s) else np.nan

fidelity_rows=[]
for (gen, seed), sdf in synthetic_logs.items():
    fidelity_rows.append({
        'generator': gen, 'seed': seed,
        'activity_kl_divergence': activity_kl(real_train, sdf),
        'trace_variant_jaccard': trace_jaccard(real_train, sdf),
        'duration_wasserstein_hours': duration_wasserstein(real_train, sdf),
        'real_cases': real_train[CASE_COL].nunique(),
        'synthetic_cases': sdf[CASE_COL].nunique(),
    })
fidelity_df = pd.DataFrame(fidelity_rows)
fidelity_df.to_csv(OUT_DIR/'fidelity_by_seed.csv', index=False)
display(fidelity_df)

In [ ]:
# ============================================================
# 7. Utility: TSTR AUC
# ============================================================
def get_feature_columns(case_df):
    exclude = {CASE_COL, 'dataset', 'trace_variant', 'case_start', 'case_end', 'subject_id'} | set(TARGETS)
    # Exclude full-trace outcome leakage. Use early prefix features only.
    exclude |= {'n_events_full', 'duration_hours_full', 'last_activity'}
    return [c for c in case_df.columns if c not in exclude]

REAL_FEATURE_COLS = get_feature_columns(real_train_cases)
print('Real feature columns:', REAL_FEATURE_COLS)

def make_model():
    if HAS_XGB:
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=2)
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced_subsample')

def build_pipeline(df, feature_cols):
    X = df[feature_cols]
    cat_cols = [c for c in feature_cols if X[c].dtype == 'object']
    num_cols = [c for c in feature_cols if c not in cat_cols]
    pre = ColumnTransformer([
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ])
    return Pipeline([('pre', pre), ('model', make_model())])

def eval_tstr(train_cases, test_cases, target, feature_cols, train_name):
    train = train_cases.dropna(subset=[target]).copy()
    test = test_cases.dropna(subset=[target]).copy()
    res = {'train_name': train_name, 'target': target, 'train_n': len(train), 'test_n': len(test), 'train_pos_rate': train[target].mean() if len(train) else np.nan, 'test_pos_rate': test[target].mean() if len(test) else np.nan, 'auc': np.nan, 'accuracy': np.nan, 'macro_f1': np.nan, 'status': 'ok'}
    if len(train) < 20 or len(test) < 20:
        res['status'] = 'too_few_cases_or_missing_label'; return res
    if train[target].nunique() < 2:
        res['status'] = 'train_single_class'; return res
    if test[target].nunique() < 2:
        res['status'] = 'test_single_class'; return res
    model = build_pipeline(train, feature_cols)
    model.fit(train[feature_cols], train[target].astype(int))
    proba = model.predict_proba(test[feature_cols])[:,1]
    pred = (proba >= 0.5).astype(int)
    res.update({'auc': float(roc_auc_score(test[target].astype(int), proba)), 'accuracy': float(accuracy_score(test[target].astype(int), pred)), 'macro_f1': float(f1_score(test[target].astype(int), pred, average='macro'))})
    return res

# Train-on-Real baseline
baseline_rows=[]
for target in TARGETS:
    baseline_rows.append(eval_tstr(real_train_cases, real_test_cases, target, REAL_FEATURE_COLS, 'Train_on_Real'))
baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(OUT_DIR/'utility_train_on_real_baseline.csv', index=False)
display(baseline_df)

utility_rows=[]
for (gen, seed), syn_cases in syn_case_tables.items():
    feature_cols = [c for c in REAL_FEATURE_COLS if c in syn_cases.columns]
    for target in TARGETS:
        res = eval_tstr(syn_cases, real_test_cases, target, feature_cols, f'{gen}_seed{seed}')
        res.update({'generator': gen, 'seed': seed, 'n_features': len(feature_cols)})
        base_auc = baseline_df.loc[baseline_df.target==target, 'auc'].iloc[0]
        res['train_on_real_auc'] = base_auc
        res['utility_gap'] = base_auc - res['auc'] if pd.notna(base_auc) and pd.notna(res['auc']) else np.nan
        utility_rows.append(res)
utility_df = pd.DataFrame(utility_rows)
utility_df.to_csv(OUT_DIR/'utility_tstr_by_seed.csv', index=False)
display(utility_df)

In [ ]:
# ============================================================
# 8. Privacy: ML-Leaks-style shadow generator MIA
# ============================================================
def case_seq_table(df):
    rows=[]
    for cid, g in df.sort_values([CASE_COL,TIME_COL]).groupby(CASE_COL):
        acts = tuple(g[ACT_COL].astype(str).tolist())
        t = pd.to_datetime(g[TIME_COL], errors='coerce')
        dur = (t.max()-t.min()).total_seconds()/3600.0 if t.notna().sum()>=2 else 0.0
        rows.append({CASE_COL: cid, 'seq': acts, 'length': len(acts), 'duration': dur})
    return pd.DataFrame(rows)

def train_shadow_generator(member_df):
    c = case_seq_table(member_df)
    return {'variants': c['seq'].tolist(), 'durations': c['duration'].to_numpy(dtype=float)}

def generate_shadow_synthetic(model, n_cases, seed):
    rng = np.random.default_rng(seed)
    variants = model['variants']; durs = model['durations']
    out=[]
    for i in range(n_cases):
        seq = variants[int(rng.integers(0, len(variants)))] if variants else ('MISSING',)
        dur = float(rng.choice(durs)) if len(durs) else 0.0
        offsets = np.linspace(0, max(dur,0), len(seq)) if len(seq)>1 else [0]
        start = pd.Timestamp('2200-01-01') + pd.Timedelta(days=i)
        cid=f'shadow_syn_{i:06d}'
        for j, act in enumerate(seq):
            out.append({CASE_COL: cid, ACT_COL: act, TIME_COL: start + pd.Timedelta(hours=float(offsets[j]))})
    return pd.DataFrame(out)

def seq_edit_norm(a,b):
    m,n=len(a),len(b)
    if m==0 and n==0: return 0.0
    dp=list(range(n+1))
    for i in range(1,m+1):
        prev, dp[0]=dp[0], i
        for j in range(1,n+1):
            cur=dp[j]; cost=0 if a[i-1]==b[j-1] else 1
            dp[j]=min(dp[j]+1, dp[j-1]+1, prev+cost); prev=cur
    return dp[n]/max(m,n,1)

def act_jacc(a,b):
    A,B=set(a),set(b)
    return len(A&B)/len(A|B) if len(A|B) else 1.0

def mia_features(candidate_df, synthetic_df, max_syn=3000):
    cand = case_seq_table(candidate_df)
    syn = case_seq_table(synthetic_df)
    if len(syn)>max_syn: syn = syn.sample(max_syn, random_state=RANDOM_STATE)
    syn_seqs=syn['seq'].tolist(); syn_lens=syn['length'].to_numpy(float); syn_durs=syn['duration'].to_numpy(float); syn_set=set(syn_seqs)
    rows=[]
    for _, r in cand.iterrows():
        seq,L,D = r['seq'], float(r['length']), float(r['duration'])
        len_diff=np.abs(syn_lens-L)
        idx=np.argsort(len_diff)[:min(100,len(syn_seqs))]
        edits=[seq_edit_norm(seq, syn_seqs[i]) for i in idx]
        jacs=[act_jacc(seq, syn_seqs[i]) for i in idx]
        dd=[abs(D-syn_durs[i]) for i in idx]
        rows.append({CASE_COL:r[CASE_COL], 'exact_variant_match':int(seq in syn_set), 'min_edit_distance':float(np.min(edits)) if edits else np.nan, 'mean_top_edit_distance':float(np.mean(sorted(edits)[:10])) if edits else np.nan, 'max_activity_jaccard':float(np.max(jacs)) if jacs else np.nan, 'min_length_diff':float(np.min(len_diff)) if len(len_diff) else np.nan, 'min_duration_diff':float(np.min(dd)) if dd else np.nan, 'candidate_length':L, 'candidate_duration':D})
    return pd.DataFrame(rows)

def fit_attack(X, y):
    cols=[c for c in X.columns if c != CASE_COL]
    pipe=Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),('mlp',MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=RANDOM_STATE, early_stopping=True))])
    pipe.fit(X[cols], y)
    return pipe, cols

def run_mia(target_syn_df, seed):
    if real_val is not None and real_val[CASE_COL].nunique() >= 50:
        ids=real_val[CASE_COL].drop_duplicates().to_numpy()
        m_ids, nm_ids = train_test_split(ids, test_size=0.5, random_state=seed)
        shadow_m = real_val[real_val[CASE_COL].isin(m_ids)]
        shadow_nm = real_val[real_val[CASE_COL].isin(nm_ids)]
    else:
        ids=real_train[CASE_COL].drop_duplicates().to_numpy()
        m_ids, nm_ids = train_test_split(ids, test_size=0.5, random_state=seed)
        shadow_m = real_train[real_train[CASE_COL].isin(m_ids)]
        shadow_nm = real_train[real_train[CASE_COL].isin(nm_ids)]
    shadow_model = train_shadow_generator(shadow_m)
    shadow_syn = generate_shadow_synthetic(shadow_model, min(N_SHADOW_SYNTH_CASES, max(100, shadow_m[CASE_COL].nunique())), seed)
    fm = mia_features(shadow_m, shadow_syn); fnm = mia_features(shadow_nm, shadow_syn)
    fm['membership']=1; fnm['membership']=0
    attack_df=pd.concat([fm,fnm], ignore_index=True)
    X=attack_df.drop(columns=['membership']); y=attack_df['membership'].astype(int)
    model, cols = fit_attack(X,y)
    tm = mia_features(real_train, target_syn_df); tnm = mia_features(real_test, target_syn_df)
    tm['membership']=1; tnm['membership']=0
    target_df=pd.concat([tm,tnm], ignore_index=True)
    scores=model.predict_proba(target_df[cols])[:,1]
    ytrue=target_df['membership'].astype(int).to_numpy(); pred=(scores>=0.5).astype(int)
    return {'mia_auc':float(roc_auc_score(ytrue,scores)), 'mia_accuracy':float(accuracy_score(ytrue,pred)), 'mia_macro_f1':float(f1_score(ytrue,pred,average='macro')), 'attack_train_n':len(attack_df), 'target_eval_n':len(target_df), 'target_member_n':int((ytrue==1).sum()), 'target_nonmember_n':int((ytrue==0).sum())}, target_df.assign(mia_score=scores)

mia_rows=[]
for (gen, seed), sdf in synthetic_logs.items():
    print('MIA:', gen, seed)
    try:
        res, scores = run_mia(sdf, int(seed))
        res.update({'generator':gen, 'seed':seed, 'status':'ok'})
        scores.to_csv(OUT_DIR/f'mia_scores_{gen}_seed{seed}.csv', index=False)
    except Exception as e:
        res={'generator':gen, 'seed':seed, 'mia_auc':np.nan, 'status':str(e)}
        print(' failed:', e)
    mia_rows.append(res)
mia_df=pd.DataFrame(mia_rows)
mia_df.to_csv(OUT_DIR/'privacy_mia_by_seed.csv', index=False)
display(mia_df)

In [ ]:
# ============================================================
# 9. Aggregate and export tables
# ============================================================
fid_agg = fidelity_df.groupby('generator').agg(activity_kl_mean=('activity_kl_divergence','mean'), activity_kl_std=('activity_kl_divergence','std'), trace_jaccard_mean=('trace_variant_jaccard','mean'), trace_jaccard_std=('trace_variant_jaccard','std'), duration_wasserstein_mean=('duration_wasserstein_hours','mean'), duration_wasserstein_std=('duration_wasserstein_hours','std'), n_seeds=('seed','count')).reset_index()
util_agg = utility_df.groupby(['generator','target']).agg(tstr_auc_mean=('auc','mean'), tstr_auc_std=('auc','std'), utility_gap_mean=('utility_gap','mean'), train_n_mean=('train_n','mean'), status_values=('status', lambda x: ';'.join(sorted(set(map(str,x)))))).reset_index()
mia_agg = mia_df.groupby('generator').agg(mia_auc_mean=('mia_auc','mean'), mia_auc_std=('mia_auc','std'), mia_acc_mean=('mia_accuracy','mean'), n_mia_seeds=('seed','count')).reset_index()

main = util_agg.merge(mia_agg, on='generator', how='left').merge(fid_agg, on='generator', how='left')
main.to_csv(OUT_DIR/'main_results_long.csv', index=False)
display(main)

prof_rows=[]
for gen in ['ProcessGAN','PALSYN','Rule_based']:
    row={'generator':gen}
    for target, kr in TARGETS.items():
        sub=main[(main.generator==gen)&(main.target==target)]
        if len(sub)==0 or pd.isna(sub.tstr_auc_mean.iloc[0]):
            cell='NA'
        else:
            cell=f'{sub.tstr_auc_mean.iloc[0]:.3f} / {sub.mia_auc_mean.iloc[0]:.3f}'
        row[kr]=cell
    prof_rows.append(row)
prof_df=pd.DataFrame(prof_rows)
prof_df.to_csv(OUT_DIR/'professor_3x3_auc_mia_table.csv', index=False)
display(prof_df)

fid_agg.to_csv(OUT_DIR/'fidelity_aggregate_by_generator.csv', index=False)
util_agg.to_csv(OUT_DIR/'utility_aggregate_by_generator_target.csv', index=False)
mia_agg.to_csv(OUT_DIR/'privacy_mia_aggregate_by_generator.csv', index=False)

In [ ]:
# ============================================================
# 10. Validity checks
# ============================================================
checks=[]
for gen in sorted({k[0] for k in syn_case_tables}):
    cases=pd.concat([v for (g,s),v in syn_case_tables.items() if g==gen], ignore_index=True)
    for target in TARGETS:
        n=int(cases[target].notna().sum()) if target in cases.columns else 0
        rate=float(cases[target].mean()) if n>0 else np.nan
        checks.append({'generator':gen, 'target':target, 'synthetic_labeled_cases':n, 'positive_rate':rate})
check_df=pd.DataFrame(checks)
check_df.to_csv(OUT_DIR/'label_availability_check.csv', index=False)
display(check_df)

print('Interpretation guide')
print('- Fidelity: KL and Wasserstein lower is better; Jaccard higher is better.')
print('- Utility: TSTR AUC higher is better; Utility Gap lower is better.')
print('- Privacy: MIA AUC close to 0.5 means random guessing; AUC > 0.6 indicates vulnerability.')
print('- NA in the 3x3 table means the synthetic log does not contain labels needed for that clinical target.')
zip_path = shutil.make_archive(str(OUT_DIR), 'zip', OUT_DIR)
print('Created:', zip_path)